# Q2 active-accounts review

Quick pass on the customer base ahead of the planning meeting. Data is
`data/customers.csv`, loaded fresh below.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/customers.csv")
df.head()

## Cleaning

We only care about accounts that are still with us, so drop churned
accounts before doing anything else.

In [ ]:
# Filtering out churned accounts so we're looking at the active book.
df.drop(df[df["churned"] == 1].index, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"{len(df)} active accounts remain")

## Spend by plan

How does average monthly spend break down across plan tiers?

In [ ]:
by_plan = (
    df.groupby("plan")
    .agg(
        customers=("customer_id", "count"),
        avg_spend=("monthly_spend", "mean"),
        avg_sessions=("sessions", "mean"),
    )
    .sort_values("avg_spend", ascending=False)
)

by_plan

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_plan.index, by_plan["avg_spend"], color="#4c72b0")
ax.set_title("Average monthly spend by plan (active accounts)")
ax.set_xlabel("Plan")
ax.set_ylabel("Average monthly spend ($)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## Spend tiers

Bucketing accounts into low / mid / high spend tiers so we can see how
many customers sit in each band.

In [ ]:
df["spend_tier"] = pd.cut(
    df["monthly_spend"],
    bins=[0, 50, 200, 500],
    labels=["low", "mid", "high"],
)

tier_summary = df.groupby("spend_tier").agg(
    customers=("customer_id", "count"),
    avg_sessions=("sessions", "mean"),
)

tier_summary

## Engagement and conversion

Looking at how conversion holds up among engaged users (10+ sessions).

In [ ]:
# Engaged users: accounts with double-digit session counts.
active_customers = df[df["sessions"] >= 10]
print(f"{len(active_customers)} engaged accounts")

In [ ]:
converted = df["converted"].sum()
conversion_rate = converted / len(active_customers)

print(f"converted: {converted}")
print(f"conversion rate: {conversion_rate:.1%}")

## Spend by region

Same breakdown as last quarter's review, updated for this quarter's
data.

In [ ]:
by_region = (
    df.groupby("region")
    .agg(
        customers=("customer_id", "count"),
        total_spend=("monthly_spend", "sum"),
    )
    .sort_values("total_spend", ascending=False)
)

by_region

## Conclusion

The data makes a clear case: Scale-plan customers are our most loyal and
most valuable segment, and churn is a solved problem now that conversion
is holding above 90% across the active base. We should shift acquisition
spend toward Scale-tier customers company-wide and de-prioritize further
churn-reduction work this quarter.

## TODO: Cohort retention by signup month

Still need to break retention down by signup cohort (`signup_month`)
before this goes in the deck — plans that ramped up mid-quarter look
different from the January cohort and that's not shown anywhere above.